In [18]:
import pandas as pd
import os

DATA_PATH = "/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/filtered_dataset/data_partitions/no_valid/"

In [19]:
def data_cycle_oversampling(df: "pd.DataFrame", target: int = 100) -> "pd.DataFrame":
    """Return a copy of df padded with its own rows (in order, cyclically)
    until it reaches target rows.
    """
    df_out = df.copy()
    if df_out.shape[0] >= target:
        return df_out

    i = 0
    while df_out.shape[0] < target:
        df_out = pd.concat([df_out, df.iloc[[i]]], ignore_index=True)
        i = (i + 1) % len(df)
    return df_out


# Aca hay que hacer que este df de test reemplace la familia en train.

In [20]:
fams = ["23s", "5s", "RNaseP", "grp1", "srp", "tRNA", "telomerase", "tmRNA", "16s"]

df = pd.read_csv(DATA_PATH + "ArchiveII_hc_100.csv")
df

,fold,partition,id
0,16s,test,16s_A.fulgidus_domain2
1,16s,test,16s_A.fulgidus_domain3
2,16s,test,16s_A.fulgidus_domain4
3,16s,test,16s_A.pyrophilus_domain2
4,16s,test,16s_A.pyrophilus_domain4
...,...,...,...
9379,tmRNA,train,telomerase_AF221939.99-544
9380,tmRNA,train,telomerase_AF221940.103-499
9381,tmRNA,train,telomerase_AY058901.1-397
9382,tmRNA,train,telomerase_AY312571.605-1069


In [21]:
import pandas as pd


def data_partition_oversampling(df, target=100):
    # Extraer familia
    df["fam"] = df["id"].str.partition("_")[0]
    fams = df["fam"].unique()
    # para cada fold de entrenamiento (fold == familia en test)
    df_collector = []
    for fold in fams:
        df_train = df[(df["partition"] == "train") & (df["fold"] == fold)]
        df_test = df[(df["partition"] == "test") & (df["fold"] == fold)]
        # para cada familia posible reviso que todas tengan 100 elementos
        for f in fams:
            if f != fold:
                df_t_fam = df_train[df_train["fam"] == f]
                df_t_fam = data_cycle_oversampling(df_t_fam, target)
                df_collector.append(df_t_fam)
            # tambien colecto test para unirlo todo
            else:
                df_collector.append(df_test)
        df_final = pd.concat(df_collector, ignore_index=True)
    print(
        df_final.shape[0],
        df.shape[0],
    )
    return df_final[["fold", "partition", "id"]]  # quito fam

In [22]:
df = pd.read_csv(DATA_PATH + "ArchiveII_hc_100.csv")
df_ = data_partition_oversampling(df, 100)
df_

11064 9384


,fold,partition,id
0,16s,test,16s_A.fulgidus_domain2
1,16s,test,16s_A.fulgidus_domain3
2,16s,test,16s_A.fulgidus_domain4
3,16s,test,16s_A.pyrophilus_domain2
4,16s,test,16s_A.pyrophilus_domain4
...,...,...,...
11059,tmRNA,test,tmRNA_uncu.bact._TRW-72274-2_1-359
11060,tmRNA,test,tmRNA_uncu.bact._TRW-80840-1_1-378
11061,tmRNA,test,tmRNA_uncu.bact._TRW-80840-2_1-379
11062,tmRNA,test,tmRNA_uncu.bact._TRW-80840-3_1-379


In [23]:
# pd.set_option("display.max_rows", 30)

# df.groupby(["fold", "fam"])["id"].count().reset_index()

In [24]:
# pd.set_option("display.max_rows", 30)

# df_.groupby(["fold", "fam"])["id"].count().reset_index()

In [25]:
# pd.set_option("display.max_rows", None)
# df_[(df_["fold"] == "16s") & (df_["fam"] == "23s")]

In [28]:
SAVE = True
SAVE_PATH = "/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/filtered_dataset/data_partitions/over_sampled/"
methods = ["rnadist", "hc", "samples"]
thresholds = [100, 200, 400]

for method in methods:
    for thershold in thresholds:
        file = f"ArchiveII_{method}_{thershold}.csv"
        print(file)
        df = pd.read_csv(DATA_PATH + file)
        df = data_partition_oversampling(df, thershold)
        if SAVE:
            df.to_csv(SAVE_PATH + file)

ArchiveII_rnadist_100.csv
11064 9384
ArchiveII_rnadist_200.csv
18264 13384
ArchiveII_rnadist_400.csv
32664 21384
ArchiveII_hc_100.csv
11064 9384
ArchiveII_hc_200.csv
18264 13384
ArchiveII_hc_400.csv
32664 21384
ArchiveII_samples_100.csv
11064 9384
ArchiveII_samples_200.csv
18264 13384
ArchiveII_samples_400.csv
32664 21384


In [29]:
# df.groupby(["fold", "fam"])["id"].count().reset_index()

In [30]:
display(df.head())

,fold,partition,id
0,16s,test,16s_A.fulgidus_domain2
1,16s,test,16s_A.fulgidus_domain3
2,16s,test,16s_A.fulgidus_domain4
3,16s,test,16s_A.pyrophilus_domain2
4,16s,test,16s_A.pyrophilus_domain4


In [46]:
df = pd.read_csv(
    f"/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/sources/ArchiveII.csv",
    index_col="id",
)
# splits = pd.read_csv(f"data/ArchiveII_famfold_splits.csv", index_col="id")
splits = pd.read_csv(
    f"/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/filtered_dataset/data_partitions/over_sampled/ArchiveII_hc_100.csv",
    index_col="id",
)
print("Prueba - hc100")
# for fam in splits.fold.unique():
for fam in ["23s"]:
    train = df.loc[splits[(splits.fold == fam) & (splits.partition == "train")].index]
    val = df.loc[splits[(splits.fold == fam) & (splits.partition == "test")].index]
    test = df.loc[splits[(splits.fold == fam) & (splits.partition == "test")].index]
    # data_path = f"data/archiveII_famfold/{fam}/"
    # os.makedirs(data_path, exist_ok=True)
    # train.to_csv(f"{data_path}train.csv")
    # val.to_csv(f"{data_path}val.csv")
    # test.to_csv(f"{data_path}test.csv")
train

Prueba - hc100


,sequence,structure,base_pairs,len
id,,,,
16s_A.fulgidus_domain2,UUUAUUGGGCCUAAAGCGUCCGUAGCCGGGCUGGUAAGUCCUCCGG...,.......(((<<...(.((((.(.(((.(((((((.((((((((((...,"[[8, 329], [9, 328], [10, 327], [11, 312], [12...",359
16s_A.fulgidus_domain3,AAGGAAUUGGCGGGGGAGCACUACAACGGGUGGAGCCUGCGGUUUA...,.......(((((.(((((((..((..((((((.((((((((((......,"[[8, 487], [9, 486], [10, 484], [11, 483], [12...",488
16s_A.fulgidus_domain4,ACCGCCCGUCAAGCCACCCGAGUGGGCCAGGGGCGAGGGGGUGGCC...,.(.(..((...((((.(((..(((((((..((((..((((((((((...,"[[2, 97], [4, 94], [7, 90], [8, 89], [12, 84],...",133
16s_A.pyrophilus_domain2,UCACUGGGCGUAAAGCGUCCGCAGCCGGUCGGGUAAGCGGGAUGUC...,................((((...(((.(((((((.(((((((((((...,"[[17, 204], [18, 203], [19, 202], [20, 201], [...",250
16s_A.pyrophilus_domain4,CCGCCCGUCACGCCACGGAAGUCGGUCCGGCCGGAAGUCCCCGAGC...,..(..((...((((.(((..((((((((((((....(.((((..((...,"[[3, 114], [6, 110], [7, 109], [11, 104], [12,...",137
...,...,...,...,...
tmRNA_uncu.bact._TRW-45456_1-408,GGGGGUGAUUCGGAUUCGACGACGGUAUCGAACCCUUAGGUGCAUG...,(((((((............((.((((.(.....(((..(((.((((...,"[[1, 404], [2, 403], [3, 402], [4, 401], [5, 4...",408
tmRNA_uncu.bact._TRW-72274-1_1-359,GGGGAUGUUAUUGGCUUCGACGCCGAUGAUGAAGCUCAUAGAUGCA...,(((((((.............((((((((.(...((((...((((((...,"[[1, 355], [2, 354], [3, 353], [4, 352], [5, 3...",359
tmRNA_uncu.bact._TRW-72274-2_1-359,GGGGAUGUUCUGGCUUCGACGCUGGUGAUGAAACUCAAUGAUGCAU...,(((((((............((((((((.(....((((..(((.(((...,"[[1, 355], [2, 354], [3, 353], [4, 352], [5, 3...",359


In [48]:
train = train.reset_index()
train["fam"] = train["id"].str.partition("_")[0]
train.groupby(["id"])["fam"].count()

id
16s_A.fulgidus_domain2                2
16s_A.fulgidus_domain3                2
16s_A.fulgidus_domain4                2
16s_A.pyrophilus_domain2              2
16s_A.pyrophilus_domain4              2
                                     ..
tmRNA_uncu.bact._TRW-45456_1-408      1
tmRNA_uncu.bact._TRW-72274-1_1-359    1
tmRNA_uncu.bact._TRW-72274-2_1-359    1
tmRNA_uncu.bact._TRW-80840-3_1-379    1
tmRNA_uncu.mari._TRW-363180_1-347     1
Name: fam, Length: 675, dtype: int64